
# Local Autonomous Knowledge AI (MacBook)

Features:
- Web search ingestion
- PDF paper reading
- YouTube transcript learning
- FAISS vector database
- Local LLM via Ollama
- ChatGPT‑style UI

Architecture

Internet → Crawl / Import → Chunk → Embedding → Vector DB → Retrieval → Local LLM


## Install Dependencies (run once)

In [29]:

pip install requests beautifulsoup4 sentence-transformers faiss-cpu ollama youtube-transcript-api pdfminer.six gradio duckduckgo-search


Note: you may need to restart the kernel to use updated packages.


## Imports

In [30]:

import requests
from bs4 import BeautifulSoup
import numpy as np
import faiss
import ollama
import gradio as gr

from sentence_transformers import SentenceTransformer
from youtube_transcript_api import YouTubeTranscriptApi
from pdfminer.high_level import extract_text
from duckduckgo_search import DDGS


## Load Embedding Model

In [31]:

embed_model = SentenceTransformer("all-MiniLM-L6-v2")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2507.30it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Data Storage

In [32]:

documents = []
metadatas = []


## Web Search + Crawl

In [33]:

def web_search(query, n=5):
    urls = []
    with DDGS() as ddgs:
        for r in ddgs.text(query, max_results=n):
            urls.append(r["href"])
    return urls

def crawl(url):
    try:
        r = requests.get(url, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")
        text = " ".join([p.get_text() for p in soup.find_all("p")])
        return text
    except:
        return ""


## Import Web Data

In [ ]:

def ingest_web(query):

    urls = web_search(query)

    for url in urls:
        text = crawl(url)
        if len(text) > 200:
            documents.append(text)
            metadatas.append({"source": url})
    
    documents.append("""
    Artificial intelligence research is rapidly evolving.
    Recent models include multimodal systems and reasoning models.
    """)

    print("Documents:", len(documents))


## Import PDF Papers

In [35]:

def ingest_pdf(path):

    text = extract_text(path)

    documents.append(text)
    metadatas.append({"source": path})

    print("PDF added.")


## Import YouTube Transcript

In [36]:

def ingest_youtube(video_id):

    transcript = YouTubeTranscriptApi.get_transcript(video_id)

    text = " ".join([x["text"] for x in transcript])

    documents.append(text)
    metadatas.append({"source": "youtube:" + video_id})

    print("YouTube transcript added.")


## Chunk Documents

In [37]:

def chunk_text(text, size=500):

    chunks = []

    for i in range(0, len(text), size):
        chunks.append(text[i:i+size])

    return chunks


## Build Vector Database

In [38]:

def build_vector_db():

    global index
    all_chunks = []

    for doc in documents:
        all_chunks.extend(chunk_text(doc))

    embeddings = embed_model.encode(all_chunks)
    embeddings = np.array(embeddings)
    print("chunks:", len(all_chunks))
    dim = embeddings.shape[1]

    index = faiss.IndexFlatL2(dim)
    index.add(embeddings)

    print("Vector DB size:", index.ntotal)

    return all_chunks


## Semantic Search

In [39]:

def search(query, chunks, k=3):

    q = embed_model.encode([query])
    q = np.array(q)

    D, I = index.search(q, k)

    results = [chunks[i] for i in I[0]]

    return results


## Ask Local LLM

In [40]:

def ask_llm(question, chunks):

    context = search(question, chunks)

    prompt = f'''
Context:
{context}

Question:
{question}
'''

    response = ollama.chat(
        model="mistral",
        messages=[{"role":"user","content":prompt}]
    )

    return response["message"]["content"]


## Build Chat Interface

In [41]:

def build_chat(chunks):

    def chat_fn(message, history):

        answer = ask_llm(message, chunks)

        history.append((message, answer))

        return history, history

    with gr.Blocks() as demo:

        chatbot = gr.Chatbot()

        msg = gr.Textbox()

        msg.submit(chat_fn, [msg, chatbot], [chatbot, chatbot])

    demo.launch()


## Example Pipeline

In [42]:

# 1. collect knowledge

ingest_web("latest AI research")

# optional
# ingest_pdf("paper.pdf")
# ingest_youtube("VIDEO_ID")

# 2. build DB
print("documents:", len(documents))
chunks = build_vector_db()

# 3. launch chat UI

build_chat(chunks)


/var/folders/zb/qq5xqfcj7xj54fm8bprnb1080000gn/T/ipykernel_10829/3549336960.py:3: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


Documents: 0
documents: 0
chunks: 0


IndexError: tuple index out of range